# Configure Application Insights for Microsoft Foundry

Application Insights captures **application-level telemetry** from your Foundry agents — every model call, tool invocation, trace span, and exception. This is the data source for the KQL demos in this section.

> **Diagnostic Settings vs. Application Insights:** Diagnostic settings route platform-level logs from the Azure resource itself. Application Insights captures what your running agent code does — the `AppRequests`, `AppDependencies`, `AppTraces`, and `AppExceptions` tables that the KQL demos query all come from Application Insights.

**What you will learn:**
- Create an Application Insights resource linked to your Log Analytics workspace
- Connect Application Insights to your Foundry project
- Enable OpenTelemetry tracing in Python agent code
- Run a test model call and verify telemetry appears in Application Insights

**Prerequisites:**
- Azure CLI installed and signed in (`az login`)
- A Log Analytics workspace (any existing workspace, or create one via the diagnostic-settings lab first)
- A deployed model in your Foundry project (e.g., `gpt-5.1`)
- **Contributor** role on the resource group

In [ ]:
%pip install python-dotenv azure-ai-projects azure-identity azure-monitor-opentelemetry openai opentelemetry-instrumentation-openai-v2 -q

## Configuration

Copy `.env.example` to `.env` and fill in your values:

| Variable | Where to find it |
|---|---|
| `RESOURCE_GROUP` | Azure Portal → Resource groups |
| `FOUNDRY_PROJECT_ENDPOINT` | ai.azure.com → your project → Overview → Foundry project endpoint |
| `MODEL_DEPLOYMENT_NAME` | Your deployed model name (e.g., `gpt-5.1`) |
| `LOG_ANALYTICS_WORKSPACE_NAME` | Your existing Log Analytics workspace name |
| `APP_INSIGHTS_NAME` | Name for your Application Insights resource (created if it does not exist) |
| `LOCATION` | Azure region matching your Foundry resources |

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

load_dotenv(override=True)

RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL = os.getenv("MODEL_DEPLOYMENT_NAME")
WORKSPACE_NAME = os.getenv("LOG_ANALYTICS_WORKSPACE_NAME")
APP_INSIGHTS_NAME = os.getenv("APP_INSIGHTS_NAME")
LOCATION = os.getenv("LOCATION", "eastus2")

# Verify Azure CLI login
result = subprocess.run(["az", "account", "show", "-o", "json"],
                        capture_output=True, text=True)
if result.returncode == 0:
    acct = json.loads(result.stdout)
    print(f"✅ Signed in as {acct['user']['name']}")
else:
    print("❌ Not signed in. Run 'az login' in your terminal first.")

print(f"\nResource Group:  {RESOURCE_GROUP}")
print(f"Project:         {ENDPOINT}")
print(f"App Insights:    {APP_INSIGHTS_NAME}")
print(f"Model:           {MODEL}")

## Step 1: Create an Application Insights Resource

Application Insights stores telemetry in a Log Analytics workspace. We create it linked to the same workspace used for diagnostic settings, so all Foundry telemetry lands in one place.

In [ ]:
# Get the Log Analytics workspace ID
result = subprocess.run(
    ["az", "monitor", "log-analytics", "workspace", "show",
     "--resource-group", RESOURCE_GROUP,
     "--workspace-name", WORKSPACE_NAME,
     "--query", "id", "-o", "tsv"],
    capture_output=True, text=True
)
WORKSPACE_ID = result.stdout.strip()

if not WORKSPACE_ID:
    print("❌ Log Analytics workspace not found. Create one first, or point LOG_ANALYTICS_WORKSPACE_NAME at an existing workspace.")
else:
    print(f"✅ Log Analytics workspace: {WORKSPACE_NAME}")

In [ ]:
# Check if App Insights already exists; create if not
result = subprocess.run(
    ["az", "monitor", "app-insights", "component", "show",
     "--app", APP_INSIGHTS_NAME,
     "--resource-group", RESOURCE_GROUP,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    app_insights = json.loads(result.stdout)
    print(f"\u2705 Application Insights already exists: {app_insights['name']}")
else:
    print(f"Creating Application Insights '{APP_INSIGHTS_NAME}'...")
    result = subprocess.run(
        ["az", "monitor", "app-insights", "component", "create",
         "--app", APP_INSIGHTS_NAME,
         "--resource-group", RESOURCE_GROUP,
         "--location", LOCATION,
         "--workspace", WORKSPACE_ID,
         "--application-type", "web",
         "-o", "json"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        app_insights = json.loads(result.stdout)
        print(f"\u2705 Created Application Insights: {app_insights['name']}")
    else:
        raise RuntimeError(f"Failed to create App Insights:\n{result.stderr}")

APP_INSIGHTS_ID = app_insights["id"]
APP_INSIGHTS_CONN_STRING = app_insights["connectionString"]
print(f"\nResource ID:\n  {APP_INSIGHTS_ID}")
print(f"\nConnection String (first 80 chars):\n  {APP_INSIGHTS_CONN_STRING[:80]}...")

## Step 2: Connect Application Insights to Your Foundry Project

In the new Microsoft Foundry, observability is configured at the **project** level. The older approach of setting an `application-insights` property on an Azure Machine Learning hub no longer applies — a Foundry project is not a hub-based Azure ML workspace.

Connect the Application Insights resource you just created to your project in the Foundry portal: open your project → **Tracing** → **Connect Application Insights** → select your resource. This is what lets the Foundry portal's **Tracing** view render your agent traces.

The cell below uses the SDK to read back the connection string your project is wired to — a quick way to confirm the connection from code.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# The SDK can read back the Application Insights connection string the project is wired to.
project_client = AIProjectClient(endpoint=ENDPOINT, credential=DefaultAzureCredential())
try:
    project_conn = project_client.telemetry.get_application_insights_connection_string()
    print("✅ Your project reports an Application Insights connection (portal-connected).")
except Exception:
    project_conn = None
    print("ℹ️  No project-level Application Insights connection detected yet.")
    print("   Connect this resource under Project → Tracing to see traces in the Foundry portal.")

# Send telemetry to the resource created in Step 1, so this lab and your KQL queries
# all read from one consistent workspace.
TELEMETRY_CONN_STRING = APP_INSIGHTS_CONN_STRING
print(f"   Telemetry destination: {APP_INSIGHTS_NAME}")

## Step 3: Enable OpenTelemetry Tracing in Python

Two pieces wire up tracing:

1. `configure_azure_monitor()` points the OpenTelemetry exporter at your Application Insights resource.
2. The OpenAI instrumentation captures each model call as a dependency span, tagging it with the `gen_ai.*` attributes (model name, token counts, finish reasons) that the KQL demos query.

In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

configure_azure_monitor(connection_string=TELEMETRY_CONN_STRING)
OpenAIInstrumentor().instrument()

print("✅ OpenTelemetry tracing configured")
print("   Model calls will be traced to Application Insights with gen_ai.* attributes")

## Step 4: Run a Test Agent Call

This cell makes a simple chat completion call through the Foundry SDK. The OpenTelemetry instrumentation captures the entire operation — the request, the model dependency call, and the response — and sends it to Application Insights.

In [ ]:
openai_client = project_client.get_openai_client()

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant. Answer in one short sentence per item."},
        {"role": "user", "content": "Name three benefits of monitoring AI agents in production. Keep it brief."},
    ],
)

print(f"Model response:\n{response.choices[0].message.content}")
print(f"\n✅ Test call complete — telemetry is being sent to Application Insights")

## Step 5: Verify Telemetry in Application Insights

Telemetry typically takes **2-5 minutes** to appear in Application Insights. The query below checks for recent data. If it returns empty results, wait a few minutes and run it again.

> You can also verify in the Azure portal: Application Insights → Transaction search → filter by the last 30 minutes.

In [ ]:
import time

print("Waiting 60 seconds for telemetry ingestion...")
time.sleep(60)

# Resolve the workspace GUID so we can query the same App* tables the KQL demos use.
ws = subprocess.run(
    ["az", "monitor", "log-analytics", "workspace", "show",
     "--resource-group", RESOURCE_GROUP,
     "--workspace-name", WORKSPACE_NAME,
     "--query", "customerId", "-o", "tsv"],
    capture_output=True, text=True
)
WORKSPACE_GUID = ws.stdout.strip()

query = (
    "AppDependencies "
    "| where TimeGenerated > ago(30m) "
    "| where Properties has 'gen_ai.system' "
    "| extend Model = tostring(Properties['gen_ai.response.model']) "
    "| project TimeGenerated, Name, Model, DurationMs, Success "
    "| take 5"
)

result = subprocess.run(
    ["az", "monitor", "log-analytics", "query",
     "--workspace", WORKSPACE_GUID,
     "--analytics-query", query,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    rows = json.loads(result.stdout)
    if rows:
        print(f"✅ Found {len(rows)} model-call record(s):\n")
        for r in rows:
            print(f"  {r.get('TimeGenerated', '')}  {r.get('Name', '')}  "
                  f"model={r.get('Model', '')}  {r.get('DurationMs', '')}ms  success={r.get('Success', '')}")
    else:
        print("No telemetry found yet. This is normal — wait 2-5 minutes and re-run this cell.")
else:
    print(f"Query failed:\n{result.stderr}")
    print("\nIf the resource was just created, wait a few minutes for initial setup to complete.")

## Summary

You created an Application Insights resource, connected it to your Foundry project, enabled OpenTelemetry tracing, and verified that agent telemetry is flowing.

**What is flowing now:**
- `AppRequests` — incoming requests to your agent
- `AppDependencies` — outbound calls (model completions, tool invocations) with `gen_ai.*` properties
- `AppTraces` — log messages and trace spans from agent operations
- `AppExceptions` — errors and stack traces

**The `gen_ai.*` properties include:**
- `gen_ai.system` — the AI provider (e.g., `openai`)
- `gen_ai.request.model` / `gen_ai.response.model` — which model was called
- `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` — token counts
- `gen_ai.response.finish_reasons` — why the model stopped generating

**Next:** The KQL demos use these tables and properties to build real monitoring queries — from basic exploration to production dashboards.